# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yousefelshazly/FlyRankMachineLearning/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

Note: I engineered a new target label (refresh_opportunity) instead of using trend_direction directly, because a simple decline flag doesn't distinguish between pages where a refresh would help and pages that are declining for reasons outside our control — low base traffic, dying topics, or unmeasurable search volume.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Answer:
Scoring because I want to score the pages and identify which ones actually need a content refresh. We do not use classification because a binary yes/no label doesn't help a content team with limited time decide which pages to review first — a score lets them prioritize.

In [ ]:
!git clone https://github.com/yousefelshazly/FlyRankMachineLearning.git
%cd FlyRankMachineLearning
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.columns.tolist())
print(df.head(2))

Cloning into 'FlyRankMachineLearning'...
remote: Enumerating objects: 124, done.
remote: Counting objects: 100% (124/124), done.
remote: Compressing objects: 100% (81/81), done.
remote: Total 124 (delta 37), reused 92 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (124/124), 1.85 MiB | 9.97 MiB/s, done.
Resolving deltas: 100% (37/37), done.
/content/FlyRankMachineLearning/FlyRankMachineLearning
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 

**The next section is exploring the data and understnading what columns will be useful for our ranking**

In [ ]:

import os
for root, dirs, files in os.walk("."):
    for f in files:
        if f.endswith(".csv"):
            print(os.path.join(root, f))
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.columns.tolist())
print(df.head(2))

./data/raw/content_refresh_anonymized.csv
./outputs/refresh_queue_sample.csv
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']
             content_id          client_id  search_volume  competition  \
0  content_304f48230142  client_f369cb89fc           10.0         0.67   
1  c

In [ ]:
print(df['search_volume'].describe())
print(df['search_volume'].value_counts().head(10))

count    27532.000000
mean       158.882391
std       1518.270825
min          0.000000
25%          0.000000
50%         10.000000
75%         20.000000
max      74000.000000
Name: search_volume, dtype: float64
search_volume
0.0      11081
10.0      7311
20.0      2290
30.0      1224
40.0       753
50.0       722
70.0       640
90.0       462
110.0      359
140.0      344
Name: count, dtype: int64


In [ ]:
zero_sv = df[df['search_volume'] == 0]
print(zero_sv['trend_direction'].value_counts())
print(zero_sv['impressions_90d'].describe())

trend_direction
down      7008
stable    1997
up        1430
new        344
flat       302
Name: count, dtype: int64
count     11081.000000
mean       5919.118942
std       16415.733533
min           1.000000
25%         128.000000
50%         998.000000
75%        4975.000000
max      497727.000000
Name: impressions_90d, dtype: float64


In [ ]:
print(df[['ctr', 'avg_position', 'impressions_last_30d',
          'clicks_last_30d', 'clicks_prev_30d']].describe())

                ctr  avg_position  impressions_last_30d  clicks_last_30d  \
count  30000.000000   30000.00000          30000.000000     30000.000000   
mean       0.510733      16.34238           1429.058733         4.933867   
std        3.279162      15.21679           5643.852081        23.929393   
min        0.000000       0.00000              0.000000         0.000000   
25%        0.000000       6.20000             10.000000         0.000000   
50%        0.070000      10.80000            139.000000         0.000000   
75%        0.290000      22.30000            768.000000         2.000000   
max      100.000000     245.00000         238796.000000      1176.000000   

       clicks_prev_30d  
count     30000.000000  
mean          5.435100  
std          28.358673  
min           0.000000  
25%           0.000000  
50%           0.000000  
75%           2.000000  
max        1627.000000  


In [ ]:
print((df['ctr'] == 0).sum())
print((df['clicks_last_30d'] == 0).sum())
print((df['clicks_prev_30d'] == 0).sum())

13212
18365
18241


In [ ]:
print(df['trend_direction'].value_counts())
print()
# check if is_declining_label exists
print('is_declining_label' in df.columns)
print()
# look at trend_pct describe
print(df['trend_pct'].describe())

trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

False

count    26612.000000
mean        -4.785969
std        473.861780
min       -100.000000
25%        -62.600000
50%        -33.500000
75%          0.000000
max      44900.000000
Name: trend_pct, dtype: float64


In [ ]:
thresholds = [10, 50, 100, 200, 500]
for t in thresholds:
    count = (df['impressions_last_30d'] >= t).sum()
    pct = (df['impressions_last_30d'] >= t).mean() * 100
    print(f"threshold {t}: {count} pages ({pct:.1f}%)")

threshold 10: 22531 pages (75.1%)
threshold 50: 18673 pages (62.2%)
threshold 100: 16299 pages (54.3%)
threshold 200: 13448 pages (44.8%)
threshold 500: 9272 pages (30.9%)


In [ ]:
declining = df[(df['trend_direction'] == 'down') & (df['impressions_last_30d'] >= 200)]
print(declining['avg_position'].describe())

count    6986.000000
mean       14.964286
std        11.127864
min         0.600000
25%         6.400000
50%        11.000000
75%        21.500000
max        77.300000
Name: avg_position, dtype: float64


In [ ]:
df['refresh_opportunity'] = (
    (df['trend_direction'] == 'down') &
    (df['impressions_last_30d'] >= 200) &
    (df['avg_position'] > 10)
).astype(int)

print(df['refresh_opportunity'].value_counts())
print(f"Refresh opportunities: {df['refresh_opportunity'].sum()} out of {len(df)} pages")
print(f"That's {df['refresh_opportunity'].mean()*100:.1f}% of all pages")

refresh_opportunity
0    26250
1     3750
Name: count, dtype: int64
Refresh opportunities: 3750 out of 30000 pages
That's 12.5% of all pages


In [ ]:
missed = df[(df['trend_direction'] == 'down') &
            (df['impressions_last_30d'] >= 200) &
            (df['avg_position'] <= 10)]
print(f"Declining pages with good impressions but ranking on page 1: {len(missed)}")
print(missed['trend_pct'].describe())

Declining pages with good impressions but ranking on page 1: 3236
count    3236.000000
mean      -45.370797
std        16.943379
min       -97.700000
25%       -57.000000
50%       -43.200000
75%       -31.000000
max       -20.000000
Name: trend_pct, dtype: float64


In [ ]:
df['refresh_opportunity'] = (
    (df['trend_direction'] == 'down') &
    (df['impressions_last_30d'] >= 200) &
    ((df['avg_position'] > 10) | (df['trend_pct'] <= -43.2))
).astype(int)

print(df['refresh_opportunity'].value_counts())
print(f"Refresh opportunities: {df['refresh_opportunity'].sum()} out of {len(df)} pages")
print(f"That's {df['refresh_opportunity'].mean()*100:.1f}% of all pages")

refresh_opportunity
0    24627
1     5373
Name: count, dtype: int64
Refresh opportunities: 5373 out of 30000 pages
That's 17.9% of all pages


In [ ]:
df['impressions_dropped'] = df['impressions_prev_30d'] - df['impressions_last_30d']
declining = df[(df['trend_direction'] == 'down') & (df['impressions_last_30d'] >= 200)]
print(declining['impressions_dropped'].describe())

count     6986.000000
mean      1884.281134
std       4087.818815
min         54.000000
25%        336.000000
50%        743.000000
75%       1795.000000
max      97995.000000
Name: impressions_dropped, dtype: float64


In [ ]:
thresholds = [200, 300, 500, 750, 1000]
for t in thresholds:
    count = ((df['trend_direction'] == 'down') &
             (df['impressions_last_30d'] >= 200) &
             (df['impressions_dropped'] >= t)).sum()
    print(f"Drop >= {t}: {count} pages")

Drop >= 200: 6133 pages
Drop >= 300: 5462 pages
Drop >= 500: 4370 pages
Drop >= 750: 3472 pages
Drop >= 1000: 2824 pages


In [ ]:
df['impressions_dropped'] = df['impressions_prev_30d'] - df['impressions_last_30d']

df['refresh_opportunity'] = (
    (df['trend_direction'] == 'down') &
    (df['impressions_last_30d'] >= 200) &
    (df['impressions_dropped'] >= 750)
).astype(int)

print(df['refresh_opportunity'].value_counts())
print(f"Refresh opportunities: {df['refresh_opportunity'].sum()} out of {len(df)} pages")
print(f"That's {df['refresh_opportunity'].mean()*100:.1f}% of all pages")

refresh_opportunity
0    26528
1     3472
Name: count, dtype: int64
Refresh opportunities: 3472 out of 30000 pages
That's 11.6% of all pages


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Answer:
The target is refresh_opportunity — a defined rule, not an observed outcome. A page is labeled 1 if it is trending down, had at least 200 impressions in the last 30 days, and lost at least 750 impressions compared to the previous 30 days. The model predicts the probability of a page meeting this definition — that probability becomes the score used to rank pages by review priority. Because this label is rule-based and not a measured future outcome, the model can only surface pages that match this definition — it cannot guarantee that refreshing a flagged page will recover its traffic.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.



## 3. Success metric

*One metric you can defend. What number means 'good'?*

Answer:
The success metric is Precision@50 — precision measured on the top 50 ranked pages only. This reflects how the content team actually works: they start from the top of the list and work down, so the model needs to be precise where it matters most. A score of 0.8 means 40 out of the top 50 flagged pages are genuine refresh opportunities — 10 false positives is an acceptable tradeoff for a team with limited review time. The baseline from week 1 was 0.740, so 0.8 is a meaningful but realistic target.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

Answer:
One row = one page. The IDs are there to locate the page after scoring — without them the list is useless. Impressions over the last 30 days and the drop from the previous period tell us how much visibility the page has lost. Average position tells us where the page ranks on Google — a page on position 70 is far from page one and needs attention. Trend direction and trend percentage confirm the page is declining. The refresh opportunity label confirms this page belongs in our lane and has the potential to recover in Google search rankings.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
lane_df = df[df['refresh_opportunity'] == 1][['content_id', 'client_id', 'impressions_last_30d',
                                               'impressions_dropped', 'avg_position',
                                               'trend_direction', 'trend_pct',
                                               'refresh_opportunity']].reset_index(drop=True)
print(f"Unit of analysis: one row = one page")
print(f"Total pages in lane: {len(lane_df)}")
lane_df.head()


Unit of analysis: one row = one page
Total pages in lane: 3472


,content_id,client_id,impressions_last_30d,impressions_dropped,avg_position,trend_direction,trend_pct,refresh_opportunity
0,content_a1fb4e703a9e,client_4e07408562,2501,3414,20.3,down,-57.7,1
1,content_9aa793d4d895,client_7f2253d7e2,2382,3707,36.5,down,-60.9,1
2,content_d99b7a2d90ca,client_3fdba35f04,4211,2241,44.0,down,-34.7,1
3,content_5e6c160719bc,client_6208ef0f77,5696,8132,46.0,down,-58.8,1
4,content_78bd1d4a1d4d,client_6208ef0f77,2422,1553,8.9,down,-39.1,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*
Answer:
A fixed rule like trend_direction == "down" is not enough on its own. A page can show a large percentage drop but have so little traffic that the decline is meaningless — 10 impressions to 8 is not worth a content team's time. A keyword can be losing traction entirely, meaning a refresh won't help regardless of the decline. Position, impression volume, and rate of drop all interact in ways that a single if-statement can't capture. ML combines these signals together and learns which combinations actually indicate a genuine refresh opportunity — producing a score that ranks pages by real urgency, not just whether one threshold was crossed.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.